## Adding Planning to CrewAI Agents

Planning enables agents to break down complex tasks into manageable steps. For our Customer Support Agent, this feature ensures systematic handling of customer inquiries, utilizing tools and structured workflows.

In [1]:
from crewai_tools import SerperDevTool

from dotenv import load_dotenv

load_dotenv()

# Instantiate the SerperDevTool for internet searches
search_tool = SerperDevTool()

In [2]:
import yaml

# Load agents from YAML file
with open('planning_agent.yaml', 'r') as file:
    agent_config = yaml.safe_load(file)

# Load tasks from YAML file
with open('planning_task.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

print(agent_config)
print("="*50)
print(task_config)

{'customer_support_agent': {'role': 'Customer Support Specialist\n', 'goal': "Assist customers with inquiries and process return requests by providing accurate information based on the company's policies. You can use the search tool to look up the cost to fix the damage to any items.\n", 'backstory': 'You are a dedicated customer support specialist with a focus on efficiency and courtesy. You collaborate with the Technical Support Agent to handle technical inquiries and ensure customer satisfaction.'}}
{'support_task': {'description': 'Handle the following support request from a user: {customer_inquiry}\\n\\n NOTE: If you need to assess damage, you can use the internet search tool to do so.\n', 'expected_output': 'An email response to the users support request.\n', 'agent': 'customer_support_agent'}}


In [ ]:
task_config["support_task"]

{'description': 'Handle the following support request from a user: {customer_inquiry}\\n\\n If you need to assess the cost of the damage, you can use the internet search tool to do so.\n',
 'expected_output': 'An email response to the users support request.\n',
 'agent': 'customer_support_agent'}

In [3]:
from crewai.tools import tool

@tool("retrieve_return_policy")
def retrieve_return_policy(query: str) -> str:
    """
    Returns the company's return policy.

    Args:
        query (str): Customer's inquiry about the return policy as a python string
    """
    return (
        "Our return policy states that if the cost to fix any damage to the item is estimated to be LESS THAN $300, "
        "you have a receipt of purchase, and you bought it within the last 60 days, "
        "you can return it and receive store credit."
    )

@tool("retrieve_shipping_policy")
def retrieve_shipping_policy() -> str:
    """
    description: Returns the company's shipping policy for handling returns.
    """
    return (
        "Our shipping policy for returns is as follows: "
        "Processing and delivery of returned items typically take 5-7 business days. "
        "Customers are responsible for a flat return shipping fee of $5 unless the item is defective. "
        "Tracking numbers are provided once the return is processed and shipped."
    )

@tool("retrieve_complaint_protocol")
def retrieve_complaint_protocol() -> str:
    """
    description: Provides the company's complaint resolution process for return-related issues.
    """
    return (
        "Our complaint resolution process for return-related issues is as follows: "
        "Complaints are acknowledged within 24 hours of submission. "
        "The resolution steps include: "
        "1) Reviewing the complaint and validating the return case, "
        "2) Contacting the customer for clarification or additional details if needed, and "
        "3) Providing a resolution, such as a refund, replacement, or escalation. "
        "If unresolved within 3 business days, the case is escalated to a senior support representative for priority handling."
    )

In [4]:
from crewai import Agent, Task, Crew, Process

agent_details = agent_config["customer_support_agent"]
customer_support_agent = Agent(
    role=agent_details["role"],
    goal=agent_details["goal"],
    backstory=agent_details["backstory"],
    tools=[retrieve_return_policy, retrieve_shipping_policy, retrieve_complaint_protocol, search_tool],
    verbose=True,
)

support_task_config = task_config["support_task"]
support_task = Task(
    description=support_task_config["description"],
    expected_output=support_task_config["expected_output"],
    agent=customer_support_agent
)


# Create and execute the crew
crew = Crew(
    agents=[customer_support_agent],
    tasks=[support_task],
    process=Process.sequential,
    planning=True,
    memory=True
)

# Kick off the crew
results = await crew.kickoff_async({"customer_inquiry": "I bought a computer 1 month ago from your company and I don't like it and I want to return it. It has a slight crack on the screen. Can I still return it?"})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Handle the following support request from a user: I bought a computer 1 month ago from your company and  │
│  I don't like it and I want to return it. It has a slight crack on the screen. Can I still return it?\n\n       │
│  NOTE: If you need to assess damage, you can use the internet search tool to do so.                             │
│  1. Start by carefully identifying the customer’s core request and the key decision factors: the item is a      │
│  computer purchased 1 month ago, the customer wants a return because they do not like it, and there is a        │
│  slight crack on the screen.                                                                                    │
│  2. Use retrieve_return_policy with a query that explicitly asks about the return window, condition             │
│  requirements, and any exclusions related to physical damage or cracked screens. Capture exact policy details   │
│  that apply to products returned after 1 month and to items with screen damage.                                 │
│  3. Use retrieve_shipping_policy to verify any return shipping requirements, such as who pays for return        │
│  shipping, whether damaged items need special packaging, whether a return authorization is required, and        │
│  whether the product must be sent back in original packaging or with accessories.                               │
│  4. Use retrieve_complaint_protocol to understand the correct escalation or complaint handling process if the   │
│  customer disputes a denial, if damaged returns need review, or if the case should be routed for exception      │
│  handling.                                                                                                      │
│  5. If the policy is unclear about whether a slight crack qualifies as returnable damage, use the internet      │
│  search tool to look up the typical repair cost or severity of a cracked screen for a computer, as allowed by   │
│  the task note. Search specifically for screen crack impact, common repair implications, and whether such       │
│  damage usually affects return eligibility under standard retail policies.                                      │
│  6. Compare all findings and determine the return outcome based strictly on policy language: confirm whether 1  │
│  month is inside or outside the return window, whether “slight crack” makes the item non-returnable or subject  │
│  to partial credit/restocking/repair review, and whether the return can proceed under any special conditions.   │
│  7. Draft the email response in a professional, empathetic tone. Acknowledge the customer’s frustration, state  │
│  the policy outcome clearly, and explain the exact reason using the retrieved policy details. If the return is  │
│  allowed, provide next steps for initiating it and any shipping instructions. If the return is not allowed      │
│  because of the crack or policy timing, explain the reason and offer any available alternatives from the        │
│  complaint protocol, such as repair, replacement review, or escalation path.                                    │
│  8. Make sure the final email is concise but complete, includes the required actions or limitation, and avoids  │
│  speculation beyond the policy and search results. If appropriate, mention any documentation the customer       │
│  should provide, such as photos of the damage or proof of purchase, if the policy or protocol requires it.      │
│  9. Before finalizing, verify that the response directl

Tool retrieve_return_policy executed with result: Our return policy states that if the cost to fix any damage to the item is estimated to be LESS THAN $300, you have a receipt of purchase, and you bought it within the last 60 days, you can return it ...
Tool retrieve_shipping_policy executed with result: Our shipping policy for returns is as follows: Processing and delivery of returned items typically take 5-7 business days. Customers are responsible for a flat return shipping fee of $5 unless the ite...
Tool retrieve_complaint_protocol executed with result: Our complaint resolution process for return-related issues is as follows: Complaints are acknowledged within 24 hours of submission. The resolution steps include: 1) Reviewing the complaint and valida...
Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'typical repair cost for a slight crack on a computer screen', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Computer 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Return Request for Computer Purchased 1 Month Ago with Screen Damage                                  │
│                                                                                                                 │
│  Dear [Customer Name],                                                                                          │
│                                                                                                                 │
│  Thank you for reaching out to us regarding your recent purchase. I understand that you would like to return    │
│  the computer you bought from us one month ago and that it has a slight crack on the screen.                    │
│                                                                                                                 │
│  Our return policy allows returns for items purchased within the last 60 days, provided you have the receipt.   │
│  However, items must not be damaged to qualify for a standard return. Since your computer has a slight crack    │
│  on the screen, it is considered damaged.                                                                       │
│                                                                                                                 │
│  We do have a policy that if the cost to fix any damage is estimated to be less than $300, you may still        │
│  return the item and receive store credit. Based on typical repair costs for a cracked computer screen, which   │
│  generally range from $80 to $300 depending on the extent of the damage, it is likely your computer qualifies   │
│  for this provision.                                                                                            │
│                                                                                                                 │
│  To proceed, please provide the purchase receipt and details or photos of the crack for us to estimate the      │
│  repair cost. Once we confirm that the repair cost is under the threshold, you can initiate the return, and we  │
│  will issue store credit accordingly.                                                                           │
│                                                                                                                 │
│  Regarding return shipping, you would be responsible for a flat return shipping fee of $5, and we advise using  │
│  a tracked shipping method to ensure safe delivery. The product should be sent back with all original           │
│  packaging and accessories if possible.                                                                         │
│                                                                                                                 │
│  If there is any dispute or additional concerns about this process, we have a complaint resolution protocol     │
│  where your case can be reviewed further and escalated if needed.                                               │
│                                                                                                                 │
│  Please let us know if you would like to proceed with the return under these conditions, and we will guide you  │
│  through the next steps.                                                                                        │
│                                                        

## Building Reflective Agents

Reflection allows agents to evaluate and improve their outputs. Our Customer Support Agent will perform a sequence of tasks, including self-assessment and revision, to enhance the quality of its responses.

In [5]:
# Load agents from YAML file
with open('reflection_agent.yaml', 'r') as file:
    agent_config = yaml.safe_load(file)

# Load tasks from YAML file
with open('reflection_task.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

print(agent_config)
print("="*50)
print(task_config)

{'customer_support_agent': {'role': 'Customer Support Specialist\n', 'goal': "Assist customers with inquiries and process return requests by providing accurate information based on the company's policies. You can use the search tool to look up the cost to fix the damage to any items.\n", 'backstory': 'You are a dedicated customer support specialist with a focus on efficiency and courtesy. You collaborate with the Technical Support Agent to handle technical inquiries and ensure customer satisfaction.'}}
{'support_task': {'description': 'Handle the following support request from a user: {customer_inquiry}\\n\\n NOTE: If you need to assess damage, you can use the internet search tool to do so.\n', 'expected_output': 'An email response to the users support request.\n', 'agent': 'customer_support_agent'}, 'reflection_task': {'description': 'Evaluate the initial email draft that you wrote and come up with specific ways to improve it. Emails need to be optimized for readability, simplicity, a

In [7]:
from crewai import Agent, Task, Crew, Process

agent_details = agent_config["customer_support_agent"]
customer_support_agent = Agent(
    role=agent_details["role"],
    goal=agent_details["goal"],
    backstory=agent_details["backstory"],
    tools=[retrieve_return_policy, retrieve_shipping_policy, retrieve_complaint_protocol, search_tool],
    verbose=True,
)

support_task_config = task_config["support_task"]
support_task = Task(
    description=support_task_config["description"],
    expected_output=support_task_config["expected_output"],
    agent=customer_support_agent
)

reflection_task_config = task_config["reflection_task"]
reflection_task = Task(
    description=reflection_task_config["description"],
    expected_output=reflection_task_config["expected_output"],
    agent=customer_support_agent,
    context=[support_task]
)


final_email_task_config = task_config["final_email_task"]
final_email_task = Task(
    description=final_email_task_config["description"],
    expected_output=final_email_task_config["expected_output"],
    agent=customer_support_agent,
    context=[support_task, reflection_task]
)


# Create and execute the crew
crew = Crew(
    agents=[customer_support_agent],
    tasks=[support_task, reflection_task, final_email_task],
    process=Process.sequential,
    planning=True,
    memory=True
)

# Kick off the crew
results = await crew.kickoff_async({"customer_inquiry": "I bought a computer 1 month ago from your company and I don't like it and I want to return it. It has a slight crack on the screen. Can I still return it?"})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Handle the following support request from a user: I bought a computer 1 month ago from your company and  │
│  I don't like it and I want to return it. It has a slight crack on the screen. Can I still return it?\n\n       │
│  NOTE: If you need to assess damage, you can use the internet search tool to do so.                             │
│  1. Start by identifying the two decision points in the user’s message: the return eligibility based on         │
│  purchase timing and the impact of the cracked screen on the return request.                                    │
│  2. Use retrieve_return_policy with a focused query such as the customer’s question about returning a computer  │
│  purchased 1 month ago, to confirm the standard return window, condition requirements, and any exceptions or    │
│  restocking rules.                                                                                              │
│  3. Use retrieve_shipping_policy to verify whether damaged items have different handling rules, whether return  │
│  shipping is allowed for damaged products, and whether the item must be packaged or processed in a specific     │
│  way.                                                                                                           │
│  4. Use retrieve_complaint_protocol to understand how the company handles return-related complaints or          │
│  exceptions if the cracked screen affects eligibility.                                                          │
│  5. If the company policy does not clearly specify whether a cracked screen affects return approval, use the    │
│  internet search tool to look up typical repair impact or damage severity guidance for a slight screen crack,   │
│  so the response can be informed and cautious.                                                                  │
│  6. Synthesize the policy information carefully: determine whether the customer is within the return window,    │
│  whether a damaged computer is still eligible, and whether the crack changes the outcome from a standard        │
│  return to an exception, rejection, or alternative resolution such as replacement or complaint escalation.      │
│  7. Draft the email response in a professional and empathetic tone, directly answering whether the computer     │
│  can still be returned, and explicitly referencing the relevant policy limitations without overexplaining.      │
│  8. Include any necessary next steps the customer should take, such as providing photos, contacting support     │
│  for inspection, using the complaint process, or initiating a return authorization if allowed.                  │
│  9. Make sure the email avoids unsupported guarantees, clearly states any condition-based restrictions, and     │
│  keeps the language simple and easy to follow.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool retrieve_return_policy executed with result: Our return policy states that if the cost to fix any damage to the item is estimated to be LESS THAN $300, you have a receipt of purchase, and you bought it within the last 60 days, you can return it ...
Tool retrieve_shipping_policy executed with result: Our shipping policy for returns is as follows: Processing and delivery of returned items typically take 5-7 business days. Customers are responsible for a flat return shipping fee of $5 unless the ite...
Tool retrieve_complaint_protocol executed with result: Our complaint resolution process for return-related issues is as follows: Complaints are acknowledged within 24 hours of submission. The resolution steps include: 1) Reviewing the complaint and valida...
Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'cost to repair slight crack on computer screen', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Computer and Tablet Re

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Return Request for Computer Purchased One Month Ago with Cracked Screen                               │
│                                                                                                                 │
│  Dear [Customer Name],                                                                                          │
│                                                                                                                 │
│  Thank you for reaching out to us regarding your recent purchase.                                               │
│                                                                                                                 │
│  Based on our company policy, customers are eligible to return items purchased within the last 60 days, as      │
│  long as they have the receipt. Since you made your purchase one month ago, you are within the return window.   │
│                                                                                                                 │
│  However, I want to inform you that our return policy considers a computer with a cracked screen as a damaged   │
│  item. For returns involving damage, if the estimated cost to fix the damage is less than $300, you may still   │
│  return the item and receive store credit.                                                                      │
│                                                                                                                 │
│  Typical repair costs for a slight crack on a computer screen generally range from approximately $80 to $300    │
│  depending on the extent of the damage. Given this, your computer appears potentially eligible for return with  │
│  store credit, subject to inspection.                                                                           │
│                                                                                                                 │
│  For returns, please use a shipping method with tracking to ensure safe delivery and verification. You would    │
│  be responsible for a flat $5 return shipping fee unless the item is found defective.                           │
│                                                                                                                 │
│  If you would like to proceed, please:                                                                          │
│  1. Provide photos of the computer, especially the cracked screen.                                              │
│  2. Contact our support team for a return authorization and further inspection details.                         │
│                                                                                                                 │
│  If you have any concerns or disagree with this outcome, you can also initiate a complaint, and we will review  │
│  and escalate your case if needed.                                                                              │
│                                                                                                                 │
│  We appreciate your understanding and are here to assist you through the process.                               │
│                                                                                                                 │
│  Best regards,                                         

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Evaluate the initial email draft that you wrote and come up with specific ways to improve it. Emails     │
│  need to be optimized for readability, simplicity, and relevancy to the users support request.  NOTE: Your      │
│  feedback should only address specific sentences or paragraphs in the email and how to improve them.            │
│  1. Read the initial email draft carefully and identify each sentence or paragraph that addresses return        │
│  eligibility, the screen crack, next steps, and any policy explanation.                                         │
│  2. Evaluate whether each part is directly relevant to the user’s question and whether any sentence adds        │
│  unnecessary detail, repetition, or unclear wording.                                                            │
│  3. Check for readability issues such as long sentences, complex phrasing, multiple ideas in one paragraph, or  │
│  policy language that could be simplified.                                                                      │
│  4. For any sentence that is too vague, specific, or overcommitted, suggest a concrete improvement focused on   │
│  clarity and accuracy.                                                                                          │
│  5. For any paragraph that combines return policy, damage assessment, and next steps in one block, recommend    │
│  splitting it into shorter sections to improve scannability.                                                    │
│  6. Ensure each suggestion names the exact sentence or paragraph to improve and explains how to improve it,     │
│  without rewriting the entire email.                                                                            │
│  7. Prioritize comments that make the response more customer-friendly, concise, and aligned with the user’s     │
│  request about returning a 1-month-old computer with a slight screen crack.                                     │
│  8. Exclude broad stylistic feedback that is not tied to a specific sentence or paragraph, since the task       │
│  requires targeted improvement suggestions only.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - The opening sentence "Thank you for reaching out to us regarding your recent purchase." is friendly but      │
│  quite generic; consider making it more specific to the issue by mentioning the cracked screen or return        │
│  request to immediately address the customer's concern.                                                         │
│                                                                                                                 │
│  - The sentence "Based on our company policy, customers are eligible to return items purchased within the last  │
│  60 days, as long as they have the receipt." could be simplified to remove unnecessary formality, e.g., say     │
│  "You can return items bought within 60 days with a receipt." This increases reading ease and directness.       │
│                                                                                                                 │
│  - The sentence "Since you made your purchase one month ago, you are within the return window." is relevant     │
│  but could be combined with the return eligibility statement for better flow, or split if combined with policy  │
│  explanations to avoid multiple ideas in one paragraph.                                                         │
│                                                                                                                 │
│  - The paragraph starting with "However, I want to inform you that our return policy considers a computer with  │
│  a cracked screen as a damaged item." contains both policy explanation and damage assessment; consider          │
│  splitting it. For example, one paragraph to explain the damage classification and another to discuss repair    │
│  cost implications and eligibility.                                                                             │
│                                                                                                                 │
│  - The phrase "if the estimated cost to fix the damage is less than $300, you may still return the item and     │
│  receive store credit." should be made more definitive or clear about possibility, e.g., clarify that the       │
│  return with store credit is conditional on inspection and repair cost assessment to avoid overcommitting.      │
│                                                                                                                 │
│  - The sentence "Typical repair costs for a slight crack on a computer screen generally range from              │
│  approximately $80 to $300 depending on the extent of the damage." adds helpful context but could be            │
│  simplified for clarity by removing redundant words like "generally" and "approximately" together.              │
│                                                                                                                 │
│  - The sentence "Given this, your computer appears potentially eligible for return with store credit, subject   │
│  to inspection." could be clearer by specifying that final eligibility depends on the inspection outcome, to    │
│  manage expectations realistically.                                                                             │
│                                                                                                                 │
│  - The paragraph discussing the shipping method and ret

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Write a final revised email response based on the bulleted list of improvements mentioned on the         │
│  initial email draft.                                                                                           │
│  1. Review the bulleted list of improvements and extract the actionable changes that must be applied to the     │
│  original draft.                                                                                                │
│  2. Re-read the support request so the final email stays tightly focused on the customer’s question: whether a  │
│  1-month-old computer with a slight screen crack can still be returned.                                         │
│  3. Revise the email to improve readability by keeping paragraphs short, using simple language, and placing     │
│  the direct answer early in the response.                                                                       │
│  4. Remove any sentences that are redundant, overly formal, or not directly relevant to the return eligibility  │
│  and damage concern.                                                                                            │
│  5. Ensure the final email reflects the correct policy details gathered earlier from retrieve_return_policy,    │
│  retrieve_shipping_policy, and retrieve_complaint_protocol, without introducing unsupported claims.             │
│  6. If the draft indicates that the crack affects return eligibility, clearly communicate that in a calm and    │
│  respectful tone; if the policy allows some exception or inspection, state the required next step clearly.      │
│  7. Make the closing section practical by including only the relevant next action for the customer, such as     │
│  providing photos, contacting support, or following the return procedure.                                       │
│  8. Before finalizing, verify that the tone is empathetic, the answer is unambiguous, and the email is concise  │
│  enough to be easily read by a customer seeking a quick resolution.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool retrieve_return_policy executed with result: Our return policy states that if the cost to fix any damage to the item is estimated to be LESS THAN $300, you have a receipt of purchase, and you bought it within the last 60 days, you can return it ...
Tool retrieve_shipping_policy executed with result (from cache): Our shipping policy for returns is as follows: Processing and delivery of returned items typically take 5-7 business days. Customers are responsible for a flat return shipping fee of $5 unless the ite...
Tool retrieve_complaint_protocol executed with result (from cache): Our complaint resolution process for return-related issues is as follows: Complaints are acknowledged within 24 hours of submission. The resolution steps include: 1) Reviewing the complaint and valida...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Return Request for Computer with Cracked Screen Purchased One Month Ago                               │
│                                                                                                                 │
│  Dear [Customer Name],                                                                                          │
│                                                                                                                 │
│  Thank you for contacting us about returning your computer with a cracked screen.                               │
│                                                                                                                 │
│  You can return items bought within 60 days if you have the receipt. Since your computer was purchased one      │
│  month ago, it is still eligible for return. However, because your computer has a cracked screen, it is         │
│  classified as a damaged item under our policy.                                                                 │
│                                                                                                                 │
│  If the repair cost for the screen crack is less than $300, you may return the item and receive store credit.   │
│  Final eligibility depends on an inspection to estimate the repair cost.                                        │
│                                                                                                                 │
│  To proceed with your return, please:                                                                           │
│  1. Send us photos of the computer, especially the cracked screen.                                              │
│  2. Provide a copy of your purchase receipt.                                                                    │
│  3. Contact our support team to get a return authorization and instructions.                                    │
│                                                                                                                 │
│  For returns, please use a shipping method with tracking. You are responsible for a $5 return shipping fee      │
│  unless the item is found to be defective.                                                                      │
│                                                                                                                 │
│  If you disagree with this process or outcome, you can file a complaint, and we will review and escalate your   │
│  case as needed.                                                                                                │
│                                                                                                                 │
│  We appreciate your understanding and are here to assist you through the return process.                        │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│  [Your Name]                                                                                                    │
│  Customer Support Specialist                                                                                    │
│  [Company Name]                                        

## Human-in-the-Loop

Incorporating human oversight ensures accurate handling of complex or sensitive issues. The Customer Support Agent can request input from a human supervisor when needed.

In [ ]:
support_task_config = task_config["support_task"]
support_task = Task(
    description=support_task_config["description"],
    expected_output=support_task_config["expected_output"],
    agent=customer_support_agent,
    human_input=True
)

reflection_task_config = task_config["reflection_task"]
reflection_task = Task(
    description=reflection_task_config["description"],
    expected_output=reflection_task_config["expected_output"],
    agent=customer_support_agent,
    human_input=True
)


final_email_task_config = task_config["final_email_task"]
final_email_task = Task(
    description=final_email_task_config["description"],
    expected_output=final_email_task_config["expected_output"],
    agent=customer_support_agent
)


# Create and execute the crew
crew = Crew(
    agents=[customer_support_agent],
    tasks=[support_task, reflection_task, final_email_task],
    process=Process.sequential,
    planning=True,
    memory=True
)

# Kick off the crew
results = crew.kickoff({"customer_inquiry": "I bought a computer 1 month ago from your company and I don't like it and I want to return it. It has a slight crack on the screen. Can I still return it?"})

# Agent: Customer Support Specialist
## Task: Handle the following support request from a user: I bought a computer 1 month ago from your company and I don't like it and I want to return it. It has a slight crack on the screen. Can I still return it?\n\nIf you need to assess the cost to repair damage, you can use the internet search tool to do so.
1. Review the user's request regarding the computer return and the crack on the screen.
2. Engage the 'retrieve_return_policy' tool to understand the company's return policy. 
   - Use the tool to gather specific information related to returns, especially concerning products with damage.
3. Document the key points from the return policy regarding damage and return timelines.
4. If possible, utilize the 'Search the internet with Serper' tool to assess the average cost of repairing a cracked computer screen. 
   - Formulate a search query that includes 'cost to repair cracked computer screen'.
5. Note any relevant insights from the cost search 